# A4 — Self-Organizing Autonomy

**The highest AWP level: recursive delegation with hierarchical budget distribution.**

A4 is the culmination of the autonomy spectrum. Agents can not only spawn workers (A2)
and create tools (A3) — they can **promote workers to sub-managers** who run their own
delegation loops with their own workers.

### The Full Spectrum at a Glance

```
A0  Prescribed       │  Static DAG, single agent
A1  Adaptive          │  Multi-agent DAG, state sharing
A2  Delegating        │  Manager spawns workers dynamically
A3  Self-Tooling      │  + Workers create tools at runtime
A4  Self-Organizing   │  + Recursive delegation, sub-managers
```

### A3 vs A4 — The Key Difference

| | A3 (Self-Tooling) | A4 (Self-Organizing) |
|---|---|---|
| Hierarchy | Manager → Workers (flat) | Manager → Sub-managers → Workers (tree) |
| Delegation depth | 1 level | Multiple levels (`max_depth`) |
| Budget model | Single pool | Distributed — parent reserves budget for children |
| Observability | Optional | **Mandatory** (tracing, metrics, audit) |
| Termination | Budget limits | Budget limits + depth cap + reservation model |

### The Recursive Delegation Tree

```
                    ┌─────────────┐
                    │  MANAGER    │  Budget: 100%
                    │  (depth 0)  │
                    └──────┬──────┘
                ┌──────────┼──────────┐
                ▼          ▼          ▼
          ┌──────────┐ ┌────────┐ ┌──────────┐
          │Sub-Mgr A │ │Worker B│ │Sub-Mgr C │
          │(depth 1) │ │        │ │(depth 1) │
          │Budget:30%│ │Bdg:10% │ │Budget:30%│
          └────┬─────┘ └────────┘ └────┬─────┘
            ┌──┼──┐                  ┌─┼──┐
            ▼  ▼  ▼                  ▼ ▼  ▼
          [w1][w2][w3]             [w4][w5][w6]
          Budget: shared           Budget: shared
          from Sub-Mgr A           from Sub-Mgr C
```

### When to use A4
- Complex multi-domain projects (e.g., enterprise software with frontend, backend, infra)
- Deep research trees (coordinator → team leads → specialists)
- Tasks requiring parallel, independent workstreams managed by sub-teams
- When a single manager cannot effectively oversee all workers directly

## 1. Provider Setup

In [ ]:
PROVIDER = "openrouter"
OLLAMA_MODEL = "qwen3:1.7b"
OLLAMA_BASE_URL = "http://localhost:11434/v1"
OPENROUTER_API_KEY = ""
OPENROUTER_MODEL = "openai/gpt-5-mini"
CUSTOM_API_KEY = ""
CUSTOM_BASE_URL = ""
CUSTOM_MODEL = ""

import os

if PROVIDER == "ollama":
    os.environ["LLM_API_KEY"] = "ollama"
    os.environ["LLM_BASE_URL"] = OLLAMA_BASE_URL
    MODEL = f"ollama/{OLLAMA_MODEL}"
elif PROVIDER == "openrouter":
    key = OPENROUTER_API_KEY or os.getenv("OPENROUTER_API_KEY", "")
    if not key:
        raise ValueError("Set OPENROUTER_API_KEY above or as an environment variable")
    os.environ["LLM_API_KEY"] = key
    os.environ["LLM_BASE_URL"] = "https://openrouter.ai/api/v1"
    MODEL = OPENROUTER_MODEL
elif PROVIDER == "custom":
    key = CUSTOM_API_KEY or os.getenv("LLM_API_KEY", "")
    url = CUSTOM_BASE_URL or os.getenv("LLM_BASE_URL", "")
    if not key or not url:
        raise ValueError("Set CUSTOM_API_KEY and CUSTOM_BASE_URL")
    os.environ["LLM_API_KEY"] = key
    os.environ["LLM_BASE_URL"] = url
    MODEL = CUSTOM_MODEL
else:
    raise ValueError(f"Unknown PROVIDER '{PROVIDER}'")

os.environ["LLM_MODEL"] = MODEL
print(f"Provider: {PROVIDER}  |  Model: {MODEL}")

## 2. A4 YAML — Recursive Delegation Configuration

A4 builds on A3 with these critical additions:

```yaml
orchestration:
  engine: delegation_loop

  delegation_loop:
    manager: agents/manager

    budget:
      max_loops: 25
      max_total_workers: 50
      max_total_tokens: 5000000
      max_wall_time: 1800
      max_tool_calls: 500
      max_depth: 3                     # CRITICAL: hard cap on recursion

    # A4: budget distribution for sub-managers
    budget_distribution:
      strategy: dynamic                # or 'equal', 'weighted'

# A4 MANDATORY: observability must be enabled
observability:
  tracing:
    enabled: true                      # Required at A4
  metrics:
    enabled: true                      # Required at A4
  audit:
    enabled: true                      # Required at A4
```

### Why `max_depth` is critical

Without a depth cap, recursive delegation could create an infinite tree:
- Manager spawns sub-manager
- Sub-manager spawns sub-sub-manager
- Sub-sub-manager spawns sub-sub-sub-manager
- ... (forever)

The **budget reservation model** prevents this:
1. When a manager promotes a worker to sub-manager, it **reserves** a portion of its budget
2. The child's budget is **strictly smaller** than the parent's remaining budget
3. At each level, the available budget shrinks → recursion terminates finitely
4. `max_depth` provides an absolute hard stop

## 3. Run an A4 Workflow — Recursive Delegation

This task is complex enough that the manager should promote workers to sub-managers.
We set `max_depth=3` to allow up to 3 levels of delegation.

In [ ]:
import time
from awp.data import AgentWorkflow

TASK = (
    "Build a comprehensive technology assessment report for a fictional startup. "
    "The report must cover three independent domains, each requiring its own research team:\n"
    "\n"
    "1. BACKEND ARCHITECTURE: Evaluate microservices vs monolith for a fintech startup. "
    "   Analyze: scalability, team autonomy, operational complexity, cost.\n"
    "2. FRONTEND STRATEGY: Compare React, Vue, and Svelte for a complex dashboard. "
    "   Analyze: developer experience, performance, ecosystem maturity.\n"
    "3. INFRASTRUCTURE: Cloud provider comparison (AWS vs GCP vs Azure) for regulated fintech. "
    "   Analyze: compliance features, cost models, managed services.\n"
    "\n"
    "For each domain, produce a standalone analysis document. "
    "Then write an executive summary that synthesizes all three into a unified recommendation. "
    "Save all documents to the output directory."
)

print(f"Task: {TASK[:80]}...")
print(f"Model: {MODEL}")
print(f"Max depth: 3 (enables sub-managers)")
print(f"Budget: 25 loops, 2M tokens, 600s wall time")
print()

t0 = time.time()

result = AgentWorkflow(
    inputs={"startup_name": "FinScale", "industry": "fintech", "team_size": "15 engineers"},
    task=TASK,
    model=MODEL,

    # A2 budget (larger for recursive work)
    max_loops=25,
    max_total_tokens=2_000_000,
    max_wall_time=600,
    max_tool_calls=200,
    max_total_workers=30,

    # A4 feature: recursive delegation depth
    max_depth=3,                  # THE A4 SWITCH

    # A3 features (included in A4)
    code_mode=True,
    tool_creation=True,
    sandbox="subprocess",

    # Manager intelligence (helps with complex task decomposition)
    planning_enabled=True,
    planning_max_subtasks=10,
    strategy_switching_enabled=True,
    budget_reservation_enabled=True,
    decision_journal_enabled=True,

    verbose=True,
).run()

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s — Status: {result['status']}")

## 4. Inspect the Delegation Hierarchy

In [ ]:
import json
from pathlib import Path

meta = result["metadata"]

print("A4 Self-Organizing Results")
print("=" * 60)
print(f"  Status:       {result['status']}")
print(f"  Loops:        {meta['loops']}")
print(f"  Workers:      {meta['workers_spawned']}")
print(f"  Tool calls:   {meta['tool_calls']}")
print(f"  Tokens:       {meta['tokens_used']:,}")
print(f"  Wall time:    {meta['wall_time']:.1f}s")
print()

# Budget utilization
budgets = [
    ("Loops",      meta["loops"],          25),
    ("Workers",    meta["workers_spawned"], 30),
    ("Tokens",     meta["tokens_used"],     2_000_000),
    ("Tool calls", meta["tool_calls"],      200),
    ("Wall time",  meta["wall_time"],       600),
]

print("Budget Utilization:")
for name, used, limit in budgets:
    pct = (used / limit * 100) if limit > 0 else 0
    bar = "#" * int(pct / 5) + "." * (20 - int(pct / 5))
    used_str = f"{used:.1f}" if isinstance(used, float) else f"{used:,}"
    print(f"  {name:<12} {used_str:>10} / {limit:>10,}  [{bar}] {pct:.0f}%")

In [ ]:
# Show output artifacts
output_dir = Path(meta.get("output_dir", "/tmp/awp-none"))
output_path = output_dir / "output"

if output_path.exists():
    files = sorted(f for f in output_path.rglob("*") if f.is_file())
    total_bytes = sum(f.stat().st_size for f in files)
    print(f"Output: {len(files)} files, {total_bytes:,} bytes total")
    print()
    for f in files:
        rel = f.relative_to(output_dir)
        sz = f.stat().st_size
        print(f"  {rel} ({sz:,} bytes)")

    # Display markdown reports
    from IPython.display import display, Markdown
    for f in files:
        if f.suffix == ".md" and f.stat().st_size < 5000:
            content = f.read_text(encoding="utf-8", errors="replace")
            display(Markdown(f"---\n### {f.name}\n\n{content[:2000]}"))
else:
    print("No output directory found.")

## 5. The Termination Guarantee

A4's recursive delegation **always terminates** because of three interlocking mechanisms:

### 1. Budget Reservation Model
When a manager promotes a worker to sub-manager, it **reserves** budget:
- The sub-manager gets a **strict subset** of the parent's remaining budget
- After reservation, the parent has less budget for future iterations
- Each level of recursion shrinks the available budget

### 2. Hard Depth Cap (`max_depth`)
Even if the budget were unlimited, `max_depth` prevents infinite nesting:
```
depth 0: Manager           (can promote to sub-managers)
depth 1: Sub-managers      (can promote if depth < max_depth)
depth 2: Sub-sub-managers  (can promote if depth < max_depth)
depth 3: STOP              (max_depth=3 reached, workers only)
```

### 3. Mandatory Observability
At A4, tracing and metrics are **required**, not optional. This ensures:
- Every delegation decision is logged
- Budget consumption is tracked at every level
- Anomalies (runaway loops, stuck sub-managers) are detectable

## 6. The Complete Autonomy Spectrum

| Level | Name | Engine | Key Addition | Safety Requirement |
|-------|------|--------|-------------|-------------------|
| **A0** | Prescribed | DAG | Static graph, 1+ agents | Schema validation (R1-R32) |
| **A1** | Adaptive | DAG | Dependencies, state sharing | `state.sharing.strategy` |
| **A2** | Delegating | Delegation Loop | Manager spawns workers | **Mandatory budgets** |
| **A3** | Self-Tooling | Delegation Loop | Dynamic tool creation | Safety envelope |
| **A4** | Self-Organizing | Delegation Loop | Recursive delegation | Depth cap + observability |

### The Design Philosophy

Each level adds **one** new capability and **one** new safety requirement:

```
Capability:  static → multi-agent → dynamic → self-tooling → recursive
Safety:      schema → state rules → budgets → sandboxes   → observability
```

**More autonomy always requires more guardrails.** This is the fundamental design principle
of AWP: the protocol does not simply unlock capabilities — it demands corresponding safety
measures. An A4 workflow that lacks mandatory observability is **not compliant**, even if
the code works.

### Choosing the Right Level

| Use case | Recommended level |
|----------|------------------|
| Text classification, simple transform | A0 |
| Multi-step pipeline (plan → research → write) | A1 |
| Open-ended creative/analytical tasks | A2 |
| Tasks requiring custom tools or API integration | A3 |
| Complex multi-domain projects with team hierarchy | A4 |

**Start at the lowest level that fits your task.** Higher autonomy means more capability
but also more complexity, cost, and unpredictability.